# Jersey colour classification in football video


In [1]:
import torch
import pandas as pd
import numpy as np

from dataset import load_manifest, get_loader
from models import load_model
from extract_embeddings import get_embeddings, extract_all_models
from classification_clustering import compare_models
from visualization import interactive_embedding_view, plot_confusion_matrix

c:\Users\Denis\OneDrive\Документы\sk_sber_football\.venv\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from pathlib import Path

manifest_path = Path("dataset_v1/manifest_with_splits.csv")

if manifest_path.exists():
    df = load_manifest(manifest_path)
else:
    print("dataset_v1/manifest_with_splits.csv not found, fallback to ../dataset/manifests/*.csv")
    manifests_dir = Path("../dataset/manifests")
    parts = []
    for split_name in ["train", "val", "test"]:
        p = manifests_dir / f"{split_name}.csv"
        if not p.exists():
            continue
        part = pd.read_csv(p)
        if "split" not in part.columns:
            part["split"] = split_name
        parts.append(part)

    if not parts:
        raise FileNotFoundError(
            "No manifests found. Expected dataset_v1/manifest_with_splits.csv or ../dataset/manifests/{train,val,test}.csv"
        )

    df = pd.concat(parts, ignore_index=True)

    # Convert detection-level manifest to the format used by this notebook.
    df["crop_path"] = df["image_path"]
    df["game"] = df["source_folder"]
    df["label"] = np.where(
        df["role_name"].astype(str).str.strip().str.lower() == "goalkeeper",
        "goalkeeper",
        np.where(df["left2right"].astype(int) == 1, "team_left", "team_right"),
    )

print("dataset size:", len(df))
df.head()

dataset_v1/manifest_with_splits.csv not found, fallback to ../dataset/manifests/*.csv
dataset size: 65356


,frame_idx,image_file,player_id,jersey,role_id,role_name,x1,y1,x2,y2,...,vx,vy,left2right,source_folder,image_path,player_global_id,split,crop_path,game,label
0,1877,frame_001877.jpg,101,5,3,Left Central Back,1399,522,1475,667,...,0.125926,-0.003292,1,game_10_H1,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1_pid101,train,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1,team_left
1,1877,frame_001877.jpg,105,20,6,Left Midfielder,77,549,126,704,...,0.088889,-0.107819,1,game_10_H1,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1_pid105,train,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1,team_left
2,1877,frame_001877.jpg,106,21,11,Right Winger,185,169,223,269,...,0.155556,0.072428,1,game_10_H1,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1_pid106,train,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1,team_left
3,1877,frame_001877.jpg,107,22,13,Right Back,285,133,312,189,...,-0.078189,-0.099588,1,game_10_H1,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1_pid107,train,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1,team_left
4,1877,frame_001877.jpg,110,99,7,Right Midfielder,1242,152,1305,260,...,0.177778,-0.071605,1,game_10_H1,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1_pid110,train,c:\Users\Denis\OneDrive\документы\sk_sber_foot...,game_10_H1,team_left


In [4]:
df["game"].unique()[:20]

<StringArray>
['game_10_H1', 'game_10_H2', 'game_11_H1', 'game_11_H2', 'game_12_H1',
 'game_12_H2', 'game_13_H1', 'game_13_H2', 'game_14_H1', 'game_14_H2',
 'game_15_H1', 'game_15_H2', 'game_16_H1', 'game_16_H2', 'game_17_H1',
 'game_17_H2', 'game_19_H1', 'game_19_H2',  'game_1_H1',  'game_1_H2']
Length: 20, dtype: str

## game_12


In [5]:
base_game = "game_12"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 722 H2: 697 Total: 1419


In [6]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_right    682
team_left     641
goalkeeper     96
Name: count, dtype: int64


In [7]:
embeddings = extract_all_models(
    df_match=df_match,
    game_id=base_game,
    device=device,
    model_names=["osnet", "dino", "fastreid", "prtreid"],
    force_recompute=True
)


  модель: osnet


c:\Users\Denis\OneDrive\Документы\sk_sber_football\.venv\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = tor

Successfully loaded imagenet pretrained weights from "C:\Users\Denis/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
computing embeddings


  0%|          | 0/12 [00:00<?, ?it/s]c:\Users\Denis\OneDrive\Документы\sk_sber_football\Football_Skoltech\models.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.device == "cuda"):
100%|██████████| 12/12 [01:51<00:00,  9.33s/it]
c:\Users\Denis\OneDrive\Документы\sk_sber_football\.venv\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


  готово: shape=(1419, 512)

  модель: dino
computing embeddings


...

building model on device cpu


C:\Users\Denis\OneDrive\Документы\sk_sber_football\prtreid\prtreid\utils\torchtools.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fpath, map_l

Successfully loaded pretrained weights from "C:\Users\Denis\OneDrive\Документы\sk_sber_football\checkpoints\job-44209669_120_model.pth.repacked.pth.tar"
** The following layers are discarded due to unmatched keys or layer size: ['backbone_appearance_feature_extractor.conv1.weight', 'backbone_appearance_feature_extractor.conv2.weight', 'backbone_appearance_feature_extractor.bn2.weight', 'backbone_appearance_feature_extractor.bn2.bias', 'backbone_appearance_feature_extractor.bn2.running_mean', 'backbone_appearance_feature_extractor.bn2.running_var', 'backbone_appearance_feature_extractor.bn2.num_batches_tracked', 'backbone_appearance_feature_extractor.layer1.3.conv1.weight', 'backbone_appearance_feature_extractor.layer1.3.bn1.weight', 'backbone_appearance_feature_extractor.layer1.3.bn1.bias', 'backbone_appearance_feature_extractor.layer1.3.bn1.running_mean', 'backbone_appearance_feature_extractor.layer1.3.bn1.running_var', 'backbone_appearance_feature_extractor.layer1.3.bn1.num_batches_t

100%|██████████| 12/12 [09:47<00:00, 48.96s/it]


  готово: shape=(1419, 512)


### Metrics

In [8]:
df_report = compare_models(embeddings)
df_report

evaluating osnet...
evaluating dino...
evaluating fastreid...
evaluating prtreid...


,accuracy,macro_f1,cluster_acc,sil_euclidean,sil_cosine,best
model,,,,,,
dino,0.9613,0.9383,0.4743,0.0902,0.1602,+
fastreid,0.9577,0.9358,0.4757,0.0500,0.0898,
osnet,0.9542,0.9233,0.7491,0.0772,0.1465,
prtreid,0.5282,0.2917,0.6779,0.1304,0.2584,


### Visualization

In [ ]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "prtreid"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1419 точек...


c:\Users\Denis\OneDrive\Документы\sk_sber_football\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...


## game_50


In [10]:
base_game = "game_40"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 765 H2: 726 Total:

 1491


In [11]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_left     726
team_right    712
goalkeeper     53
Name: count, dtype: int64


In [12]:
embeddings = extract_all_models(
    df_match=df_match,
    game_id=base_game,
    device=device,
    model_names=["osnet", "dino", "dinov2_large", "fastreid", "prtreid"],
)


  модель: osnet


c:\Users\Denis\OneDrive\Документы\sk_sber_football\.venv\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = tor

Successfully loaded imagenet pretrained weights from "C:\Users\Denis/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
computing embeddings


  0%|          | 0/12 [00:00<?, ?it/s]c:\Users\Denis\OneDrive\Документы\sk_sber_football\Football_Skoltech\models.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.device == "cuda"):
 67%|██████▋   | 8/12 [02:05<01:02, 15.65s/it]


KeyboardInterrupt: 

### Metrics

In [ ]:
df_report = compare_models(embeddings)
df_report

evaluating osnet...
evaluating dino...
evaluating dinov2_large...
evaluating fastreid...
evaluating clip...


,accuracy,macro_f1,cluster_acc,sil_euclidean,sil_cosine,best
model,,,,,,
osnet,0.9691,0.9477,0.7479,0.1122,0.2066,+
dino,0.9725,0.9320,0.7431,0.1173,0.2129,
fastreid,0.9622,0.8819,0.6605,0.0946,0.1665,
dinov2_large,0.9656,0.8292,0.4401,0.0687,0.1203,
clip,0.9588,0.7060,0.7342,0.1234,0.2196,


### Visualization

In [ ]:
viz_cols = [
    "crop_path", "game", "frame_idx", "player_id",
    "x1", "y1", "x2", "y2", "source_folder", "image_file",
]

df_viz = df_match[[c for c in viz_cols if c in df_match.columns]].reset_index(drop=True)

name = "osnet"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH